# Day 10: Testing with pytest
Objectives:
- Write unit tests and run pytest from the classroom pad.
- Use assertions and parametrize.
- Structure tests for utilities created earlier.

<!-- BEGIN BEGINNER NOTEBOOK DEEP DIVE -->
## How to use this notebook

This is the editable learner artifact for `python-10`. Read
`python/ds-60day/companion-guides/day10_testing_pytest.md` first, then work here with the **Python (ds60sqlpy)**
kernel. Restart the kernel and run from top to bottom so an earlier
hidden value cannot make later code appear correct.

For every example: (1) write a prediction, (2) run the cell,
(3) compare the exact value, type, shape, rows, or side effect with
the stated observation, and (4) explain one mismatch before moving
on. For every exercise, use its dedicated work cell and include a
real assertion or bounded inspection. The official solution stays
closed until you have a tested attempt.

The notebook is deliberately offline after course setup. Do not add
`%pip`, credentials, absolute developer paths, or shell-specific
setup here. If an import fails, use the repository doctor and the
catalog from a terminal rather than changing only this kernel.

## Core mental model

A test is a small executable example of observable behavior. Arrange
the inputs, act once, then assert the output or failure. Tests do not
prove a program has no bugs; they make chosen contracts repeatable and
reveal when later changes violate them.

Good cases cover a normal path, exact boundaries, and invalid input.
Keep each test focused enough that its name and failure identify the
broken behavior. Fixtures own reusable setup and teardown. Parametrized
tests express the same contract across several inputs without hiding
which case failed.

### Vocabulary

- **test case:** one named example with inputs and expected behavior.
- **assertion:** a claim that must be true for the test to pass.
- **fixture:** reusable setup data or a managed resource supplied to tests.
- **parametrization:** running one test contract with multiple named inputs.
- **regression:** behavior that used to work but was later broken.
- **test isolation:** the property that one test does not depend on another's state or order.

## Syntax anatomy

In `assert actual == expected`, pytest can display both values and their
difference. `with pytest.raises(ValueError, match="positive"):` states
that the indented call must raise that exact failure and optionally
match a stable message fragment. The test name should state behavior,
such as `test_safe_divide_rejects_zero_denominator`.

### Worked example 1: Test a pure calculation with plain assertions

Direct equality makes failures explain the two values. Before running the next cell, predict its final displayed
value and identify the line responsible for every intermediate.

In [ ]:
def subtotal(prices: list[float]) -> float:
    return sum(prices)


assert subtotal([2.5, 3.0]) == 5.5
assert subtotal([]) == 0
"two subtotal contracts passed"

**Expected observation:** `'two subtotal contracts passed'`. If an assertion fails, execution stops at the violated contract.

If your result differs, compare inputs and types before rerunning.
Then explain the example from input to evidence in your own words.

### Worked example 2: Represent cases as data

The same behavior can be checked across several boundaries. Predict first; then run the next cell.

In [ ]:
def is_valid_percentage(value: float) -> bool:
    return 0 <= value <= 100


cases = [(0, True), (100, True), (-0.1, False), (100.1, False)]
results = [(value, is_valid_percentage(value) == expected) for value, expected in cases]
results

**Expected observation:** `[(0, True), (100, True), (-0.1, True), (100.1, True)]`. Each tuple reports that its expectation matched.

## Debugging clinic

When evidence differs from your prediction, use this order:

1. Read the first failing assertion and reproduce that smallest case before scanning later failures.
2. Assert directly on useful values rather than wrapping them in an opaque helper returning only `True`/`False`.
3. Make fixtures create and clean up only state owned by the test.
4. If a test passes alone but fails in the suite, look for shared mutable state, order dependence, time, or randomness.

**Alternative to compare:** A doctest can suit tiny documentation examples, while pytest is clearer for fixtures, parametrization, exceptions, and larger behavior contracts.

**Boundary to test:** Empty input, exact bounds, malformed values, repeated calls, and cleanup after an exception reveal weak tests.

Do not move on merely because the cell runs. Explain which object,
branch, axis, row, or resource changed and why.

## Run pytest from the classroom pad

pytest is a separate program: it finds files named `test_*.py`, runs every
`test_*` function in them, and prints a report. A classroom Run executes one
program, so the recipe below writes the worked example's module and test file
into a temporary folder, runs pytest on that folder as a child process with
`subprocess.run`, prints pytest's report, and deletes the folder. The empty
`pytest.ini` keeps pytest from reading settings from anywhere else. Reuse the
recipe for the exercises by replacing the two source strings.

Predict the summary line before running the next cell.

In [ ]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

module_source = """def normalize_age(raw: str) -> int:
    age = int(raw)
    if age < 0:
        raise ValueError("age must be non-negative")
    return age
"""

test_source = """import pytest

from ages import normalize_age


@pytest.mark.parametrize(("raw", "expected"), [("0", 0), ("42", 42)])
def test_normalize_age(raw: str, expected: int) -> None:
    assert normalize_age(raw) == expected


def test_normalize_age_rejects_negative() -> None:
    with pytest.raises(ValueError, match="non-negative"):
        normalize_age("-1")
"""

# Write a tiny project into a temporary folder, run pytest on it in a child
# process, and delete the folder afterwards.
with tempfile.TemporaryDirectory(prefix="day10-") as folder:
    project = Path(folder)
    (project / "pytest.ini").write_text("[pytest]\n", encoding="utf-8")
    (project / "ages.py").write_text(module_source, encoding="utf-8")
    (project / "test_ages.py").write_text(test_source, encoding="utf-8")
    environment = dict(os.environ, PYTEST_DISABLE_PLUGIN_AUTOLOAD="1", PYTHONDONTWRITEBYTECODE="1")
    environment.pop("PYTEST_ADDOPTS", None)
    result = subprocess.run(
        [
            sys.executable,
            "-m",
            "pytest",
            "-q",
            "--color=no",
            "-p",
            "no:cacheprovider",
            "test_ages.py",
        ],
        cwd=project,
        env=environment,
        capture_output=True,
        text=True,
        timeout=60,
    )

print(result.stdout)
assert result.returncode == 0, result.stdout + result.stderr

**Expected observation:** three dots and `3 passed`: two parametrized cases
and the negative test. Change `match="non-negative"` to
`match="positive integer"` and rerun: the report ends with
`FAILED test_ages.py::test_normalize_age_rejects_negative` and
`1 failed, 2 passed`, and the failure explains `Regex pattern did not match`
with the expected pattern and the actual message.

### Optional local workflow

In a prepared contributor checkout (see [CONTRIBUTING](../../../CONTRIBUTING.md)),
save the two files as `ages.py` and `test_ages.py` in a scratch folder outside
the repository and run pytest on that folder from the repository root:

```bash
uv run --no-sync python -m pytest -q path/to/scratch-folder
```

## Exercises and progressive hints

Each item is a complete mini-contract. Before writing code, copy its input,
expected behavior, constraints, and verification into your work cell. A
result is not complete merely because it “looks right”; run the stated
assertion or inspection and explain what it proves.

1. Write pytest tests for `safe_divide(numerator, denominator)` and `validate_email(text)`. **Coverage:** one ordinary result, negative values, zero denominator, one accepted bounded email example, and several rejected near misses. **Constraints:** name each behavior clearly and compare observable return values directly.
   **Verify:** run the tests with the pad recipe in "Run pytest from the classroom pad" and confirm that the summary counts every case you wrote as passed.

2. Add negative tests with `pytest.raises` for the narrow exception types documented by both functions. **Constraints:** keep only the expected failing call inside each context manager and match one stable, useful message fragment.
   **Verify:** temporarily change the expected exception once to observe a meaningful failure, then restore it.

### Additional mastery practice

Test observable contracts across normal, boundary, and invalid inputs. A good failure explains which behavior changed.

Continue with five new exercises. Record each prediction before running
code; these extend rather than replace the original practice above.

3. **Prediction:** Predict how pytest reports `assert actual == expected` compared with `assert check(actual)` when the values differ.
   **Progressive hint:** Direct comparisons usually produce more useful assertion introspection.
   **Verify:** Run both deliberately failing assertions once and compare pytest's messages; record which report exposes actual/expected values directly.
4. **Tracing:** Trace fixture setup, test execution, and teardown when the test passes and when it raises.
   **Progressive hint:** A yielding fixture resumes after `yield` for cleanup in both paths.
   **Verify:** Append events from setup, test, and teardown for pass and raise cases; assert teardown is the final event in both traces.
5. **Implementation:** Write parameterized tests for a slug function covering ordinary text, extra whitespace, punctuation, and empty text.
   **Progressive hint:** Each parameter row should communicate one behavior.
   **Verify:** Confirm pytest reports a separate named case for ordinary, whitespace, punctuation, and empty inputs and that all match the written slug contract.
6. **Debugging:** Repair a test whose `pytest.raises(Exception)` would accept unrelated bugs and whose protected block contains several operations.
   **Progressive hint:** Assert a narrow exception around one operation.
   **Verify:** Prove the narrow expected error passes, then trigger an unrelated error and confirm the test fails instead of accepting it.
7. **Edge case and explanation:** Test floating-point output and `NaN` correctly. Explain why direct equality is inappropriate for each.
   **Progressive hint:** Use `pytest.approx` for tolerance and `math.isnan` for NaN.
   **Verify:** Assert one computed float passes with `pytest.approx`, assert `math.isnan(value)` is true for NaN, and record that `value == value` is false.

Before opening the reference solution, write one sentence explaining
which contract or mental model each result confirms.

### Practice 1: prediction, attempt, and evidence

**Contract reminder:** Write pytest tests for `safe_divide(numerator, denominator)` and `validate_email(text)`. **Coverage:** one ordinary result, negative values, zero denominator, one accepted bounded email example, and several rejected near misses. **Constraints:** name each behavior clearly and compare observable return values directly. **Verify:** run the tests with the pad recipe in "Run pytest from the classroom pad" and confirm that the summary counts every case you wrote as passed.

Write your prediction in the next cell before any code, then finish with the
check named in **Verify**. If it fails, keep the smallest failing input.

In [ ]:
# Practice 1: your work
# Task: Write pytest tests for `safe_divide(numerator, denominator)` and `validate_email(text)`. Coverage: one ordinary result, negative values, zero denominator, one accepted bounded email example, and several rejected near misses. Constraints: name each behavior clearly and compare observable return values directly.
# Verify: Run the tests with the pad recipe in "Run pytest from the classroom pad" and confirm that the summary counts every case you wrote as passed.
# Prediction:

# Implementation:

# Verification:


### Practice 2: prediction, attempt, and evidence

**Contract reminder:** Add negative tests with `pytest.raises` for the narrow exception types documented by both functions. **Constraints:** keep only the expected failing call inside each context manager and match one stable, useful message fragment. **Verify:** temporarily change the expected exception once to observe a meaningful failure, then restore it.

Write your prediction in the next cell before any code, then finish with the
check named in **Verify**. If it fails, keep the smallest failing input.

In [ ]:
# Practice 2: your work
# Task: Add negative tests with `pytest.raises` for the narrow exception types documented by both functions. Constraints: keep only the expected failing call inside each context manager and match one stable, useful message fragment.
# Verify: Temporarily change the expected exception once to observe a meaningful failure, then restore it.
# Prediction:

# Implementation:

# Verification:


### Practice 3: prediction, attempt, and evidence

**Contract reminder:** **Prediction:** Predict how pytest reports `assert actual == expected` compared with `assert check(actual)` when the values differ. **Progressive hint:** Direct comparisons usually produce more useful assertion introspection. **Verify:** Run both deliberately failing assertions once and compare pytest's messages; record which report exposes actual/expected values directly.

Write your prediction in the next cell before any code, then finish with the
check named in **Verify**. If it fails, keep the smallest failing input.

In [ ]:
# Practice 3: your work
# Task: Predict how pytest reports `assert actual == expected` compared with `assert check(actual)` when the values differ.
# Verify: Run both deliberately failing assertions once and compare pytest's messages; record which report exposes actual/expected values directly.
# Prediction:

# Implementation:

# Verification:


### Practice 4: prediction, attempt, and evidence

**Contract reminder:** **Tracing:** Trace fixture setup, test execution, and teardown when the test passes and when it raises. **Progressive hint:** A yielding fixture resumes after `yield` for cleanup in both paths. **Verify:** Append events from setup, test, and teardown for pass and raise cases; assert teardown is the final event in both traces.

Write your prediction in the next cell before any code, then finish with the
check named in **Verify**. If it fails, keep the smallest failing input.

In [ ]:
# Practice 4: your work
# Task: Trace fixture setup, test execution, and teardown when the test passes and when it raises.
# Verify: Append events from setup, test, and teardown for pass and raise cases; assert teardown is the final event in both traces.
# Prediction:

# Implementation:

# Verification:


### Practice 5: prediction, attempt, and evidence

**Contract reminder:** **Implementation:** Write parameterized tests for a slug function covering ordinary text, extra whitespace, punctuation, and empty text. **Progressive hint:** Each parameter row should communicate one behavior. **Verify:** Confirm pytest reports a separate named case for ordinary, whitespace, punctuation, and empty inputs and that all match the written slug contract.

Write your prediction in the next cell before any code, then finish with the
check named in **Verify**. If it fails, keep the smallest failing input.

In [ ]:
# Practice 5: your work
# Task: Write parameterized tests for a slug function covering ordinary text, extra whitespace, punctuation, and empty text.
# Verify: Confirm pytest reports a separate named case for ordinary, whitespace, punctuation, and empty inputs and that all match the written slug contract.
# Prediction:

# Implementation:

# Verification:


### Practice 6: prediction, attempt, and evidence

**Contract reminder:** **Debugging:** Repair a test whose `pytest.raises(Exception)` would accept unrelated bugs and whose protected block contains several operations. **Progressive hint:** Assert a narrow exception around one operation. **Verify:** Prove the narrow expected error passes, then trigger an unrelated error and confirm the test fails instead of accepting it.

Write your prediction in the next cell before any code, then finish with the
check named in **Verify**. If it fails, keep the smallest failing input.

In [ ]:
# Practice 6: your work
# Task: Repair a test whose `pytest.raises(Exception)` would accept unrelated bugs and whose protected block contains several operations.
# Verify: Prove the narrow expected error passes, then trigger an unrelated error and confirm the test fails instead of accepting it.
# Prediction:

# Implementation:

# Verification:


### Practice 7: prediction, attempt, and evidence

**Contract reminder:** **Edge case and explanation:** Test floating-point output and `NaN` correctly. Explain why direct equality is inappropriate for each. **Progressive hint:** Use `pytest.approx` for tolerance and `math.isnan` for NaN. **Verify:** Assert one computed float passes with `pytest.approx`, assert `math.isnan(value)` is true for NaN, and record that `value == value` is false.

Write your prediction in the next cell before any code, then finish with the
check named in **Verify**. If it fails, keep the smallest failing input.

In [ ]:
# Practice 7: your work
# Task: Test floating-point output and `NaN` correctly. Explain why direct equality is inappropriate for each.
# Verify: Assert one computed float passes with `pytest.approx`, assert `math.isnan(value)` is true for NaN, and record that `value == value` is false.
# Prediction:

# Implementation:

# Verification:
